## Basics of arcpy
- The `arcpy` Python package ships with the base Python environment that comes with ArcGIS Pro
- Python scripts should begin with `import` statements telling a user what packages are needed to run the code
- `arcpy` needs to imported just like any other Python module
- Here we import `arcpy` and check what the default workspace is in the environment settings

In [ ]:
import arcpy

### `arcpy` syntax
- `arcpy` is called
- this is followed by a dot (.) and then the toolbox module (this follows convention of toolboxes you are used to in ArcGIS Pro)
- Followed by another dot (.) and the specific `arcpy` tool name

In [ ]:
arcpy.analysis.Buffer
help(arcpy.analysis.Buffer)

`help` is useful to learn more about the tool from Python and see what the required tool parameters are (optional parameters for a tool are always in curly braces)

### Common `arcpy` toolbox modules include
- `arcpy.analysis`
- `arcpy.management`
- `arcpy.conversion`
- `arcpy.sa`


### How might we discover what available toolboxes are?

In [ ]:
arcpy.ListToolboxes()

### `arcpy` Functions
- `arcpy` not only has toolbox tools, it also has a number of useful built-in functions
- These include
   - The **Describe** function
   - **List** functions
   - **env** 
   - **Exists** to test existence of workspace, dataset or layer


### Workspace
In order to use a number of the list functions we need to first set our workspace

In [ ]:
# Define folder path and geodatabase name
out_folder_path = r"C:\Users\mwebe\OneDrive\UW Program\GISPR\Module 2" # replace with an output folder on your machine
out_name = "my_workspace.gdb"

# Create the file geodatabase workspace
arcpy.management.CreateFileGDB(out_folder_path, out_name)

# Set it as the active workspace environment
arcpy.env.workspace = f"{out_folder_path}/{out_name}"

### Explore the contents in a geopackage
- Let's look at a quick introductory worked example iterating over contents in a file geodatabase would like much the same; a geopackage is an open format similar in many ways to a file geodatabase
- We import libaries we are using first
- We use some handy code to pull from our course GitHub repo (we'll talk more about this as the course goes on) and download to a local directoy
- We define our workspace and our path
- We then use methods in arcpy to iterate over layers in our geopackage and print them out
- Notice the use of an if else clause to capture the condition of not finding anything and letting us know!

In [ ]:
import requests
from pathlib import Path
from tkinter import Tk, filedialog

RAW_URL = "https://raw.githubusercontent.com/UWC2-GISPR/modules/main/Marys.gpkg"

root = Tk()
root.withdraw()
root.attributes("-topmost", True)
dest_dir = filedialog.askdirectory(title="Choose download folder")
root.destroy()

if not dest_dir:
    raise SystemExit("No folder selected.")

dest = Path(dest_dir) / "Marys.gpkg"

with requests.get(RAW_URL, stream=True) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print(f"Saved to {dest}  ({dest.stat().st_size:,} bytes)")

In [ ]:
import arcpy
import os

# Set the workspace to the GeoPackage
arcpy.env.workspace = dest

# List all feature classes inside the GeoPackage
feature_classes = arcpy.ListFeatureClasses()

if feature_classes:
    for fc in feature_classes:
        print(f"--- Feature Class: {fc} ---")       
else:
    print("No feature classes found in the specified GeoPackage.")


### Explore your current ArcGIS Pro session with arcpy
We can also leverage `arcpy` to see what we currently have in our active ArcGIS Pro project - output will depend on the project you have open and whenther your workspace has any feature classes currently.

In [ ]:
import arcpy

# arcpy.env.workspace points to your default geodatabase or a folder
aprx = arcpy.mp.ArcGISProject("Current") # your current Pro project GDB
arcpy.env.workspace = aprx.defaultGeodatabase   

# List all feature classes in the workspace
fcs = arcpy.ListFeatureClasses()
print("Feature classes in workspace:")
for fc in fcs:
    desc = arcpy.Describe(fc)
    print(f"  {fc:<30} {desc.shapeType:<12} CRS: {desc.spatialReference.name}")

# If your workspace has no feature classes, this cell will print an empty list —
# that's fine. The key is that arcpy imported and ran without error.

### Let's load these features into our Map Contents

In [ ]:
import os
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap or aprx.listMaps()[0]
arcpy.env.overwriteOutput = True

fcs = arcpy.ListFeatureClasses()
for fc in fcs:
    m.addDataFromPath(os.path.join(arcpy.env.workspace, fc))

### Let's use the buffer operation we asked for help on in earlier cell above
We'll buffer our dams in the Mary's River Watershed by 100 meters and add to our ArcGIS Pro Contents pane and map

In [ ]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap or aprx.listMaps()[0]
arcpy.env.overwriteOutput = True

in_fc  = "main_MarysRiverDams"                        # the layer name in your Contents pane
out_fc = os.path.join(aprx.defaultGeodatabase, "Dams_buffered")

# 100 m buffer. Works even if the data are in lat/long, because Pro measures true meters there.
arcpy.analysis.PairwiseBuffer(in_fc, out_fc, "100 Meters", dissolve_option="NONE")
print(arcpy.management.GetCount(out_fc)[0], "buffers written to", out_fc)

# Add to the map once, named Cams_buffered (Pro may already have added it automatically)
lyr = next((l for l in m.listLayers() if l.name == "NWIS_buffered"), None)
if lyr is None:
    lyr = m.addDataFromPath(out_fc)
    lyr.name = "NWIS_buffered"
print(f"Layer '{lyr.name}' is in map '{m.name}'")

### Working with the `arcgis` API for Python for interacting with content in our organization ArcGIS Online portal

- First we can see how to do an anonymous connection to simply view public content without being logged into an account
- Then we can see how to log into our organization account to see content that would only be available when logged in

In [ ]:
# [Python] ArcGIS API for Python — connect to ArcGIS Online and browse items
# This works outside ArcGIS Pro, in a cloned environment, as long as arcgis is installed

from arcgis.gis import GIS

# Anonymous connection — access public content only
gis = GIS()
print(f"Connected as: {gis.properties.user.username if gis.users.me else 'anonymous'}")

# Search for a publicly available feature layer
results = gis.content.search(
    query="US States boundaries",
    item_type="Feature Layer",
    max_items=3
)

print("\nPublic feature layers found:")
for item in results:
    print(f"  {item.title[:50]:<52} | {item.type} | owner: {item.owner}")

# To load one of these as a Spatially Enabled DataFrame:
# from arcgis.features import FeatureLayer
# lyr = FeatureLayer(results[0].url)
# sdf = lyr.query().sdf   # sdf = Spatially Enabled DataFrame
# print(sdf.head())

In [ ]:
from arcgis.gis import GIS
gis = GIS("home")
gis.properties.name

### Display a hosted feature layer shared to the GISPR group in the UW Supplemental ArcGIS Online organization

In [ ]:
ITEM_ID = "9cc8b3b5cee040ebbedf9bad0ece6d15"

item = gis.content.get(ITEM_ID)
if item is None:
    raise ValueError("Item not found. Check the ID, and that Pro is signed in to the org that owns it.")

print(f"Title:    {item.title}")
print(f"Type:     {item.type}")
print(f"Owner:    {item.owner}")

for i, lyr in enumerate(getattr(item, "layers", None) or []):
    p = lyr.properties
    n = lyr.query(return_count_only=True) if p.get("type") == "Feature Layer" else "n/a"
    print(f"\nLayer {i}: {p.name}  ({p.get('geometryType', p.type)}, {n} features)")
    print("  Fields:", [f["name"] for f in p.get("fields", [])])

item   # shows the item card with its thumbnail

### We can add the item we just called to our active ArcGIS Pro session
Washington Lakes will appear in your ArcGIS Pro table of contents after you run the cell below

In [ ]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap or aprx.listMaps()[0]

if item.type == "Web Map":
    urls = [ol["url"] for ol in item.get_data().get("operationalLayers", []) if "url" in ol]
else:
    urls = [l.url for l in item.layers]
for u in urls:
    m.addDataFromPath(u)
print(f"Added {len(urls)} layer(s) to map '{m.name}'")

In [ ]:
from arcgis.gis import GIS

gis = GIS("home")
results = gis.content.search(
    query='title:"USA Major Cities" AND owner:esri_dm',
    item_type="Feature Layer",
    outside_org=True
)
item = results[0]
item